In [ ]:
# Core imports for data handling and file system navigation.
# Using pandas for dataframe operations and Path for clean file path management.

import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import torchaudio
from sklearn.model_selection import train_test_split

import os
import sys

In [ ]:
# Mounting Google Drive so the dataset and CSV metadata can be accessed directly.
# All audio files and split CSVs live inside the project directory on Drive.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Root directory for this project’s dataset.
# This is where the dementia and nodementia folders and CSV metadata files are stored. Be sure to update path to match your own.

save_path = save_path = Path('/content/drive/MyDrive/CS7357_Project/data')

In [ ]:
# Utility import for optionally inspecting audio or debugging file contents if needed.

import IPython.display
import json

def Audio(audio: np.ndarray, sr: int):
    """
    Use instead of IPython.display.Audio as a workaround for VS Code.
    `audio` is an array with shape (channels, samples) or just (samples,) for mono.
    """

    if np.ndim(audio) == 1:
        channels = [audio.tolist()]
    else:
        channels = audio.tolist()

    return IPython.display.HTML("""
        <script>
            if (!window.audioContext) {
                window.audioContext = new AudioContext();
                window.playAudio = function(audioChannels, sr) {
                    const buffer = audioContext.createBuffer(audioChannels.length, audioChannels[0].length, sr);
                    for (let [channel, data] of audioChannels.entries()) {
                        buffer.copyToChannel(Float32Array.from(data), channel);
                    }

                    const source = audioContext.createBufferSource();
                    source.buffer = buffer;
                    source.connect(audioContext.destination);
                    source.start();
                }
            }
        </script>
        <button onclick="playAudio(%s, %s)">Play</button>
    """ % (json.dumps(channels), sr))

In [ ]:
# Define separate paths for dementia and nodementia audio folders.
# Keeping these explicit makes it easier to control labeling later.

dm_path = save_path / 'dementia'
nd_path = save_path / 'nodementia'

In [ ]:
# Load metadata CSVs that contain names and dataset split information.
# These CSVs determine which speaker belongs to train or validation.

dm_df = pd.read_csv(save_path/'dementia.csv')
nd_df = pd.read_csv(save_path/'nodementia.csv')

In [ ]:
dm_df.head()

,name,dementia type,birth,death,first symptoms,URLs after symptoms,5 years,5 < 10 years,10 < 15 years,gender,ethnicity,datasplit,language,unknown 1,unkown 2,unknown 3
0,Abe Burrows,Alzheimer,1910,1985,1975.0,NaN,https://www.youtube.com/watch?v=VezbsmCriw4,NaN,NaN,male,Caucasian/White,train,NaN,NaN,NaN,NaN
1,Aileen Hernandez,Dementia,1926,2017,2012.0,https://youtu.be/x7hujcEhQuY,https://youtu.be/CshhDl-YwkY \nhttps://youtu.b...,NaN,NaN,female,Black/African American,train,NaN,NaN,NaN,NaN
2,Alan Ramsey,Dementia,1938,2020,2015.0,NaN,https://www.youtube.com/watch?v=CHeXE4c6EDI,NaN,NaN,male,Caucasian/White,train,NaN,NaN,NaN,NaN
3,Allan Burns,Lewy body,1935,2021,NaN,NaN,https://www.youtube.com/watch?v=aD3hL-kWoPc,NaN,NaN,male,Caucasian/White,train,NaN,NaN,NaN,NaN
4,Andrew Sachs,Dementia,1930,2016,2012.0,NaN,NaN,https://youtu.be/FSgKLooW1LM,https://youtu.be/3V1iFmavqG4,male,NaN,train,NaN,NaN,NaN,NaN


In [ ]:
# Utility function to split dataframe by a specific column value.
# Used to separate training and validation speakers cleanly.

def split_df(df, col, val):
    return df[df[col] == val], df[df[col] != val]
        # Returns two dataframes:
    # One matching the split value and one containing the rest.

In [ ]:
# Split dementia metadata into training and validation sets
# based on the predefined 'datasplit' column.

valid_dm, train_dm = split_df(dm_df, 'datasplit', 'valid')
test_dm, train_dm = split_df(train_dm, 'datasplit', 'test')

valid_nd, train_nd = split_df(nd_df, 'datasplit', 'valid')

In [ ]:
# Convert speaker names into lists for faster membership checks
# when scanning audio directories.

train_dmlst = train_dm['name'].tolist()
train_ndlst = train_nd['name'].tolist()
valid_dmlst = valid_dm['name'].tolist()
valid_ndlst = valid_nd['name'].tolist()

In [ ]:
print(len(train_dmlst), len(train_ndlst), len(valid_dmlst), len(valid_ndlst))

68 50 14 11


In [ ]:
# This will store structured training entries:
# Each item will contain file path and label.

data_train = []
data_valid = []
# Create normalized lists for efficient lookup
train_dmlst_norm = [name.lower().replace(' ', '') for name in train_dmlst]
valid_dmlst_norm = [name.lower().replace(' ', '') for name in valid_dmlst]

for path in tqdm(dm_path.glob('**/*.wav')):
    # Normalize person name from path for comparison
    person = str(path).split('/')[-2]
    person_norm = person.lower().replace(' ', '')

    if person_norm in train_dmlst_norm:
        try:
            s = torchaudio.load(path)
            data_train.append({ 'file': str(path).split('/')[-1].split('.')[0], 'label': 'dementia', 'path': path })
        except Exception as e:
            print(f'{path} is not a valid wav file', e)
            pass
    elif person_norm in valid_dmlst_norm:
        try:
            s = torchaudio.load(path)
            data_valid.append({ 'file': str(path).split('/')[-1].split('.')[0], 'label': 'dementia', 'path': path })
        except Exception as e:
            print(f'{path} is not a valid wav file', e)
            pass

131it [01:17,  1.69it/s]


In [ ]:
# Normalize names to avoid issues with spacing, capitalization,
# or formatting inconsistencies between CSV metadata and folder names.
train_ndlst_norm = [name.lower().replace(' ', '') for name in train_ndlst]
valid_ndlst_norm = [name.lower().replace(' ', '') for name in valid_ndlst]

for path in tqdm(nd_path.glob('**/*.wav')):
    # Normalize person name from path for comparison
    person = str(path).split('/')[-2]
    person_norm = person.lower().replace(' ', '')

    if person_norm in train_ndlst_norm:
        try:
            s = torchaudio.load(path)
            data_train.append({ 'file': str(path).split('/')[-1].split('.')[0], 'label': 'nodementia', 'path': path })
        except Exception as e:
            print(f'{path} is not a valid wav file', e)
            pass
    elif person_norm in valid_ndlst_norm:
        try:
            s = torchaudio.load(path)
            data_valid.append({ 'file': str(path).split('/')[-1].split('.')[0], 'label': 'nodementia', 'path': path })
        except Exception as e:
            print(f'{path} is not a valid wav file', e)
            pass

238it [00:44,  5.30it/s]


In [ ]:
# Convert accumulated training samples into a DataFrame
# so it can be fed into the Hugging Face Dataset pipeline.
train_df = pd.DataFrame(data_train)
valid_df = pd.DataFrame(data_valid)

train_df.head()

,file,label,path
0,ianmccaskill_15,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...
1,woodydurham_0_2,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...
2,SparkyAnderson_0,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...
3,DonLane_0,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...
4,PeterMax_10,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...


In [ ]:
valid_df.head()

,file,label,path
0,ErnieSigley_15,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...
1,JoeConley_0,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...
2,JimmyFratianno_0,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...
3,LeroneBennettJr_10,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...
4,EstelleGetty_15,dementia,/content/drive/MyDrive/CS7357_Project/data/dem...


In [ ]:
# do a quick check to confirm correct label encoding.
# Ensures binary classification labels are set as expected.

print("Labels: ", train_df.label.unique())
print(len(train_df.label.unique()))

Labels:  ['dementia' 'nodementia']
2


In [ ]:
# Verify class balance in training data.
# Important to check whether dementia and control classes are reasonably distributed.
train_df.groupby('label').count()[['path']]

,path
label,
dementia,106
nodementia,121


In [ ]:
# Confirm total number of training samples after processing.
print(f"train: {len(train_df)}")
print(f"valid: {len(valid_df)}")

train: 193
valid: 41


In [ ]:
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

train_df.to_csv(save_path / 'train_dm.csv', sep='\t', encoding='utf-8', index=False)
valid_df.to_csv(save_path / 'valid_dm.csv', sep='\t', encoding='utf-8', index=False)


In [ ]:
# Debug check to confirm audio files are being discovered correctly.
# Prints first few files to ensure directory structure matches expectations.
print(f"Files in dm_path: {list(dm_path.glob('**/*.wav'))[:5]}") # Print first 5 files found
print(f"Files in nd_path: {list(nd_path.glob('**/*.wav'))[:5]}") # Print first 5 files found

Files in dm_path: [PosixPath('/content/drive/MyDrive/CS7357_Project/data/dementia/Ian McCaskill/ianmccaskill_15.wav'), PosixPath('/content/drive/MyDrive/CS7357_Project/data/dementia/Woody Durham/woodydurham_0_2.wav'), PosixPath('/content/drive/MyDrive/CS7357_Project/data/dementia/Sparky Anderson/SparkyAnderson_0.wav'), PosixPath('/content/drive/MyDrive/CS7357_Project/data/dementia/Don Lane/DonLane_0.wav'), PosixPath('/content/drive/MyDrive/CS7357_Project/data/dementia/Peter Max/PeterMax_10.wav')]
Files in nd_path: [PosixPath('/content/drive/MyDrive/CS7357_Project/data/nodementia/Merle Haggard/MerleHaggard_1.wav'), PosixPath('/content/drive/MyDrive/CS7357_Project/data/nodementia/Merle Haggard/MerleHaggard_3.wav'), PosixPath('/content/drive/MyDrive/CS7357_Project/data/nodementia/Merle Haggard/MerleHaggard_2.wav'), PosixPath('/content/drive/MyDrive/CS7357_Project/data/nodementia/Anthony Hopkins/AnthonyHopkins_1.wav'), PosixPath('/content/drive/MyDrive/CS7357_Project/data/nodementia/Anthon